In [2]:
import numpy as np
import pydicom
import os
import cv2
import glob
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt

from skimage.metrics import structural_similarity as ssim
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from skimage.feature import hessian_matrix, hessian_matrix_eigvals
from skimage.filters import sato
from scipy.ndimage import gaussian_filter, label, binary_closing
from skimage import filters
from scipy.ndimage import binary_dilation, binary_fill_holes
from skimage.filters import gaussian


### 1. extract frames and cropping

In [52]:
def extract_frames_from_dicom(dicom_path, output_folder, input_index):
    
    input_index_li = input_index.split(',')
    dicom_data = pydicom.dcmread(dicom_path+input_index_li[0]+"\\00572087_20250103_1_"+input_index_li[-1]+".dcm")
    image_num = int(input_index_li[0])
    #print(input_index_li, input_index_li[0])

    if not os.path.exists(output_folder): 
        os.makedirs(output_folder)

    # get pixel image
    pixel_array = dicom_data.pixel_array

    # traverse all frames
    for frame_index in range(pixel_array.shape[0]):
        frame = pixel_array[frame_index]

        # if image is gray, translate to three channels (RGB)
        if len(frame.shape) == 2:
            frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

        #output_path = os.path.join(output_folder, f"sten_{image_num:04d}_{frame_index:04d}.png")
        output_path = os.path.join(output_folder, f"{frame_index}.png")
        cv2.imwrite(output_path, frame)

        #print(f"Saved {output_path}")
    

In [53]:
#def remove_border(image_path, output_path, input_index):#remove the black border of images

#    count = 0
#    input_index_li = input_index.split(',')
#    folder_path = image_path+f"{input_index_li[0]}\\output"

#    for image in glob.glob(os.path.join(folder_path, '*.png')):
    
#        img = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
    
        # 二值化处理，以便更容易识别黑色边框
#        _, thresh = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    
        # 找到轮廓
#        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
        # 找到最大的轮廓
#        if contours:
#            largest_contour = max(contours, key=cv2.contourArea)
#            x, y, w, h = cv2.boundingRect(largest_contour)
        
            # 裁剪图片
#            img_cropped = img[y:y+h, x:x+w]
        
            # 保存裁剪后的图片
#            cv2.imwrite(output_path+f"\\{count}.png", img_cropped)

#            count = count+1

In [54]:
def crop_and_resize_image(input_path, output_path, input_index):#remove the black border of images

    count = 0
    input_index_li = input_index.split(',')
    path = input_path+f"{input_index_li[0]}\\output"

    for image in glob.glob(os.path.join(path, '*.png')):
    #image -> "F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\xxx.png"
    
        with Image.open(image) as img:
            width, height = img.size

            border_width = int(width * 0.05)#remove 5% of width
            border_height = int(height * 0.05)#remove 5% of height

            cropped_img = img.crop((border_width, border_height, width - border_width, height - border_height))
        
       
            resized_img = cropped_img.resize((width, height), Image.Resampling.LANCZOS)#keep the images in the same size as before
        
        
            resized_img.save(output_path+f"\\{count}.png")

            print(output_path+f"\\{count}.png")

            count = count+1

In [55]:
#dicom_path = "D:\\00200440\\Huang Rui Cheng\\Huang Rui Cheng\\20250103102526.486000\\1\\00572087_20250103_1_13.dcm"

dicom_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\"
#input_li = ["1,13","2,18"]
input_li = ["1,13","2,18","3,110","4,14","5,16","6,111","7,115","8,112","9,114","10,11","11,12","12,15","13,17","14,19","15,113"]
for i in input_li:
    input_index_li = i.split(',')#[1,13]
    output_folder = dicom_path+f"{input_index_li[0]}\\output"
    extract_frames_from_dicom(dicom_path, output_folder, i)
    crop_and_resize_image(dicom_path,output_folder,i)

F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\0.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\1.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\2.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\3.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\4.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\5.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\6.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\7.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\8.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\9.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\output\10.png
F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.4

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0009.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0010.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0011.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0012.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0013.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0014.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0015.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0016.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0017.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0003_0018.png
Saved D:\00200440\Hu

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0015.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0016.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0017.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0018.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0019.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0020.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0021.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0022.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0023.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0005_0024.png
Saved D:\00200440\Hu

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0022.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0023.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0024.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0025.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0026.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0027.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0028.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0029.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0030.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0031.png
Saved D:\00200440\Hu

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0116.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0117.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0118.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0119.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0120.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0121.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0122.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0123.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0124.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0125.png
Saved D:\00200440\Hu

...

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0493.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0494.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0495.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0496.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0497.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0498.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0499.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0500.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0501.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0502.png
Saved D:\00200440\Hu

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0586.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0587.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0588.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0589.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0590.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0591.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0592.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0593.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0594.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0595.png
Saved D:\00200440\Hu

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0679.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0680.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0681.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0682.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0683.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0684.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0685.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0686.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0687.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0007_0688.png
Saved D:\00200440\Hu

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0011_0024.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0011_0025.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0011_0026.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0011_0027.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0011_0028.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0012_0000.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0012_0001.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0012_0002.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0012_0003.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0012_0004.png
Saved D:\00200440\Hu

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0014_0039.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0014_0040.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0014_0041.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0014_0042.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0014_0043.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0015_0000.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0015_0001.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0015_0002.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0015_0003.png
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\output\sten_0015_0004.png
Saved D:\00200440\Hu

### 2. Auto-selection of Best Frame

In [14]:
def find_min_pixel_sum_frame(image_folder):

    best_frame = None
    min_pixel_sum = float('inf')
    
    for image_path in glob.glob(os.path.join(image_folder, '*.png')):

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        

        pixel_sum = np.sum(img)
        
        if pixel_sum < min_pixel_sum:
            min_pixel_sum = pixel_sum
            best_frame = img
    
    return best_frame

def save_min_pixel_sum_frame(image_folder, output_path):
    best_frame = find_min_pixel_sum_frame(image_folder)
    
    if best_frame is not None:
        cv2.imwrite(output_path, best_frame)
        print(f"Saved frame with minimum pixel sum to {output_path}")
    else:
        print("No frames found or processed.")

for i in range(1,16):        
    input_folder = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output'
    output_path = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\sharpest_frame.png'
    save_min_pixel_sum_frame(input_folder, output_path)

Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\sharpest_frame.png
Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\2\sharpest_frame.png
Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\3\sharpest_frame.png
Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\4\sharpest_frame.png
Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\5\sharpest_frame.png
Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\6\sharpest_frame.png
Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\7\sharpest_frame.png
Saved frame with minimum pixel sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102

In [24]:
def calculate_histogram_frequency(image):
    
    histogram = cv2.calcHist([image], [0], None, [256], [0, 256])
    
    # find the greatest number of frequencies of the smallest pixel value in histogram
    frequencies = histogram.flatten()
    sorted_indices = np.argsort(frequencies)[:20]
    frequency_sum = np.sum(frequencies[sorted_indices])
    
    return frequency_sum

def find_max_frequency_frame(image_folder):
    
    best_frame = None
    max_frequency_sum = 0

    
    for image_path in glob.glob(os.path.join(image_folder, '*.png')):
       
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        
        frequency_sum = calculate_histogram_frequency(img)
        
        if frequency_sum > max_frequency_sum:
            max_frequency_sum = frequency_sum
            best_frame = img

    return best_frame

def save_max_frequency_frame(image_folder, output_path):
    best_frame = find_max_frequency_frame(image_folder)
    
    if best_frame is not None:
        cv2.imwrite(output_path, best_frame)
        print(f"Saved frame with maximum frequency sum to {output_path}")
    else:
        print("No frames found or processed.")

for i in range(1,16):        
    input_folder = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output'
    output_path = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\sharpest_frame2.png'
    save_max_frequency_frame(input_folder, output_path)

Saved frame with maximum frequency sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\sharpest_frame2.png
Saved frame with maximum frequency sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\2\sharpest_frame2.png
Saved frame with maximum frequency sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\3\sharpest_frame2.png
Saved frame with maximum frequency sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\4\sharpest_frame2.png
Saved frame with maximum frequency sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\5\sharpest_frame2.png
Saved frame with maximum frequency sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\6\sharpest_frame2.png
Saved frame with maximum frequency sum to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\7\sharpest_frame2.png
Saved frame with maximum frequency sum to F:\JJU Dataset\CAG w

In [16]:
def calculate_local_contrast(image):

    blurred = cv2.GaussianBlur(image, (5, 5), 0)
    contrast = cv2.absdiff(image, blurred)
    local_contrast = np.mean(contrast)
    return local_contrast

def find_best_contrast_frame(image_folder):

    best_frame = None
    max_contrast = 0
    
    for image_path in glob.glob(os.path.join(image_folder, '*.png')):

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        contrast = calculate_local_contrast(img)
        
        if contrast > max_contrast:
            max_contrast = contrast
            best_frame = img
    
    return best_frame

def save_best_contrast_frame(image_folder, output_path):
    best_frame = find_best_contrast_frame(image_folder)
    
    if best_frame is not None:
    
        cv2.imwrite(output_path, best_frame)
        print(f"Saved best contrast frame to {output_path}")
    else:
        print("No frames found or processed.")

for i in range(1,16):        
    input_folder = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output'
    output_path = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\sharpest_frame3.png'
    save_best_contrast_frame(input_folder, output_path)

Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\sharpest_frame3.png
Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\2\sharpest_frame3.png
Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\3\sharpest_frame3.png
Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\4\sharpest_frame3.png
Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\5\sharpest_frame3.png
Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\6\sharpest_frame3.png
Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\7\sharpest_frame3.png
Saved best contrast frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\8\sharpest_frame3.png
Saved best contrast frame to F:\

In [17]:
def calculate_brightness_contrast(image):
    
    brightness = np.mean(image)
    
    contrast = np.std(image)
    
    image_quality = brightness + contrast
    
    return image_quality

def find_best_quality_frame(image_folder):
    
    best_frame = None
    max_quality = 0
    
    for image_path in glob.glob(os.path.join(image_folder, '*.png')):
    
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        quality = calculate_brightness_contrast(img)
        
        if quality > max_quality:
            max_quality = quality
            best_frame = img

    return best_frame

def save_best_quality_frame(image_folder, output_path):
    best_frame = find_best_quality_frame(image_folder)
    
    if best_frame is not None:

        cv2.imwrite(output_path, best_frame)
        print(f"Saved best quality frame to {output_path}")
    else:
        print("No frames found or processed.")

for i in range(1,16):        
    input_folder = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output'
    output_path = f'F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\sharpest_frame4.png'
    save_best_quality_frame(input_folder, output_path)

Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\1\sharpest_frame4.png
Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\2\sharpest_frame4.png
Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\3\sharpest_frame4.png
Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\4\sharpest_frame4.png
Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\5\sharpest_frame4.png
Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\6\sharpest_frame4.png
Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\7\sharpest_frame4.png
Saved best quality frame to F:\JJU Dataset\CAG without CCTA\Huang Rui Cheng\20250103102526.486000\8\sharpest_frame4.png
Saved best quality frame to F:\JJU Datas

### 3. PCA for Vectorized Frames

In [2]:
image_folder = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\1\\output"
img_li = []

for image_path in glob.glob(os.path.join(image_folder, '*.png')):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    vec_img = img.flatten()#vectorize image matrix
    img_li.append(vec_img)
frame_vectors = np.vstack(img_li)
print(frame_vectors.shape)

(42, 262144)


In [3]:
num_vec = frame_vectors.shape[0]

pca = PCA(n_components=num_vec)
pca.fit(frame_vectors)

# principle components
components = pca.components_

height, width = img.shape[0], img.shape[1]

# transform principle components to image shape
component_images = components.reshape(num_vec, height, width)
print(component_images)

[[[ 7.74303650e-15 -3.63795738e-15 -1.42961912e-15 ... -4.79331481e-04
   -4.45953745e-04 -2.27523512e-04]
  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -3.91069082e-04
   -1.85492413e-04  1.62357665e-04]
  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -1.13131716e-04
    9.26480950e-05  3.40900645e-04]
  ...
  [ 2.13780737e-03  2.15056589e-03  2.42199445e-03 ...  1.64156968e-03
    1.70328576e-03  1.66355503e-03]
  [ 2.16877676e-03  2.12122800e-03  2.25121072e-03 ...  1.73139645e-03
    1.83589450e-03  1.95510557e-03]
  [ 2.05841102e-03  2.11644395e-03  2.18781716e-03 ...  1.93868063e-03
    2.16106625e-03  2.12905449e-03]]

 [[-3.47654359e-15  6.76449501e-16  6.34664245e-17 ... -1.17965187e-03
   -1.18184153e-03 -1.23275755e-03]
  [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -1.52363081e-03
   -1.23063015e-03 -9.89693894e-04]
  [-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 ... -1.68393133e-03
   -1.03408700e-03 -3.63860832e-04]
  ...
  [-5.65160065e-04 -5.6

In [5]:
top_n_components = 10  # top 10 components
component_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\1\\component_image"

for i in range(top_n_components):
    filename = os.path.join(component_path, f'component_{i + 1}.png')
    plt.imsave(filename, component_images[i], cmap='gray')

### 4. Demeaned and Intense Process

In [153]:
# mean extraction process
def mean_extract(image_folder, mean_path):
    img_li = []

    for image_path in glob.glob(os.path.join(image_folder, '*.png')):
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        vec_img = img.flatten()#vectorize image matrix
        img_li.append(vec_img)
    frame_vectors = np.vstack(img_li)

    num_vec = frame_vectors.shape[0]

    pca = PCA(n_components=num_vec)
    pca.fit(frame_vectors)

    # mean
    mean = pca.mean_

    height, width = img.shape[0], img.shape[1]

    # transform mean vector back to image
    mean_image = mean.reshape(1, height, width)
    bg = mean_image.squeeze()
    filename = os.path.join(mean_path, f'mean.png')
    plt.imsave(filename, bg, cmap='gray')

In [154]:
for i in range(1,16):
    image_folder = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output"
    mean_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}"
    mean_extract(image_folder,mean_path)

In [155]:
#demeaned process
def demeaned_process(img_path, mean_path, output_path):
    count = 0

    for path in glob.glob(os.path.join(img_path, '*.png')):
        img1 = cv2.imread(mean_path, cv2.IMREAD_GRAYSCALE).astype(np.int16)
        img2 = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.int16)
        #img_new = img2 - img1
        img_new = cv2.subtract(img2,img1)
        filename = os.path.join(output_path, f'{count}.png')
        cv2.imwrite(filename, img_new)#don't use plt.imsave(filename, img_new, cmap='gray'), it will normalized to [0,1] and change the initial pixel value
        count = count+1

In [156]:
for i in range(1,16):
    img_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output"
    mean_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\mean.png"
    output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\demeaned_image"
    demeaned_process(img_path, mean_path, output_path)

In [157]:
# intense process: mean (from others) + k* demeaned_image
def intense_process_forall(demeaned_path, mean_path, k, output_path, n):
    demeaned_img = cv2.imread(demeaned_path, cv2.IMREAD_GRAYSCALE).astype(np.int16)
    mean = cv2.imread(mean_path, cv2.IMREAD_GRAYSCALE).astype(np.int16)

    #demeaned_img[demeaned_img > 100] = 255
    img_new2 = mean+k*demeaned_img
    filename = os.path.join(output_path, f'intense_frame_{n}.png')
    cv2.imwrite(filename, img_new2)

In [165]:
k = 1.2
p = 1# a specific mean background, e.g., "\\20250103102526.486000\\1\\mean.png"

for m in range(1,2):
    demeaned_path_0 = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\demeaned_image"
    img_num = len(glob.glob(os.path.join(demeaned_path_0, '*.png')))
    for n in range(img_num):
        demeaned_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\demeaned_image\\{n}.png"#need to specify a best frame
        mean_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{p}\\mean.png"#need to specify a mean from other patient
        output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\intense_images"
        intense_process_forall(demeaned_path, mean_path, k, output_path, n)

#### 4.1 Variance of Image, and Use Var to drop noises

In [28]:
def calculate_variance(image_folder, num_images, image_size=(512, 512)):
    
    # Initialize an array to hold the sum of all images
    total_sum = np.zeros(image_size, dtype=np.float64)
    
    # Initialize an array to hold the sum of squares of all images
    total_sum_squares = np.zeros(image_size, dtype=np.float64)
    
    # Process each image
    full_data = []
    for i in range(num_images):
        image_path = os.path.join(image_folder, f'{i}.png')
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE).astype(np.float64)

        full_data.append(image.flatten())
        # Accumulate the sum and sum of squares
        #total_sum += image
    
    #print(len(full_data))

    variance= np.var(full_data, axis = 0)
    #print(variance.shape[0])
    
    return variance.reshape([512, 512])

In [ ]:
# Usage for Algorithm 1 (robust filter)
if __name__ == "__main__":

    for i in range(2,3):
        robust_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\robust_preprocess"
        for j in range(0, len(glob.glob(os.path.join(robust_path, '*.png')))):
        
            image_folder = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output" 
            input_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\robust_preprocess\\robust_preprocess_{j}.png"
    
            num_images = len(glob.glob(os.path.join(image_folder, '*.png')))
            variance = calculate_variance(image_folder, num_images)
            output_path_var = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}"
            filename = os.path.join(output_path_var, f'variance_{i}.png')
            cv2.imwrite(filename, variance)

            t = np.percentile(variance, 75)
            variance[variance <= t] = 0
            variance[variance > t] = 1
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            new_img = np.multiply(img, variance)
        
            output_path_new = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\variance_robust"
            filename = os.path.join(output_path_new, f'product_{i}_{j}.png')
            cv2.imwrite(filename, new_img)

In [52]:
# Usage for Edge Detector
if __name__ == "__main__":

    for i in range(2,3):
        robust_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\edge_detect_thres"
        for j in range(0, len(glob.glob(os.path.join(robust_path, '*.png')))):
        
            image_folder = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output" 
            input_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\edge_detect_thres\\edge_thres_{j}.png"
    
            num_images = len(glob.glob(os.path.join(image_folder, '*.png')))
            variance = calculate_variance(image_folder, num_images)
            output_path_var = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}"
            filename = os.path.join(output_path_var, f'variance_{i}.png')
            cv2.imwrite(filename, variance)

            t = np.percentile(variance, 65)
            variance[variance <= t] = 0
            variance[variance > t] = 1
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            new_img = np.multiply(img, variance)
        
            output_path_new = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\variance_edge"
            filename = os.path.join(output_path_new, f'product_{i}_{j}.png')
            cv2.imwrite(filename, new_img)

In [32]:
# Usage for Edge Detector
if __name__ == "__main__":

    for i in range(2,3):
        img_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\flood_fill.png"
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image_folder = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}\\output" 
    
        num_images = len(glob.glob(os.path.join(image_folder, '*.png')))
        variance = calculate_variance(image_folder, num_images)
        output_path_var = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{i}"
        filename = os.path.join(output_path_var, f'variance_{i}.png')
        cv2.imwrite(filename, variance)

        t = np.percentile(variance, 65)
        variance[variance <= t] = 0
        variance[variance > t] = 1
        new_img = np.multiply(img, variance)
        
        output_path_new = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2"
        filename = os.path.join(output_path_new, f'xxx.png')
        cv2.imwrite(filename, new_img)

In [102]:
#demeaned process for variance_robust
img_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\variance_robust"
output_demeaned_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\demeaned_variance_robust"

total_sum = np.zeros((512, 512))
for i in glob.glob(os.path.join(img_path, '*.png')):
    img = cv2.imread(i, cv2.IMREAD_GRAYSCALE)
    total_sum = img+total_sum
mean = total_sum/len(glob.glob(os.path.join(img_path, '*.png')))
filename = os.path.join(output_demeaned_path, f'var_robust_mean.png')
cv2.imwrite(filename, mean)

count = 0

for i in glob.glob(os.path.join(img_path, '*.png')):
    img = cv2.imread(i, cv2.IMREAD_GRAYSCALE)
    demeaned_img = img-mean
    filename = os.path.join(output_demeaned_path, f'demeaned_image_{count}.png')
    cv2.imwrite(filename, demeaned_img)
    count = count+1

#### 4.2 Find Intersections of All Frames and Delete Fixed Noise

In [53]:
img_folder = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\variance_edge"#variance_robust
output_common_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2"
img_li = []

for img_path in glob.glob(os.path.join(img_folder, '*.png')):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_li.append(img)


def find_common_pixels(matrix_li):
    # stack into a 3D matrix
    stacked_matrices = np.stack(matrix_li, axis=0)  # shape is (41, 512, 512)
    
    # find common pixels
    common_pixels = np.all(stacked_matrices == stacked_matrices[0], axis=0)
    
    # create a new matrix to store common pixels, and mark 0 for other pixels
    result_matrix = np.where(common_pixels, stacked_matrices[0], 0)
    
    return result_matrix

common_pixel = find_common_pixels(img_li)
print(common_pixel)
filename = os.path.join(output_common_path, f'common_pixel2.png')
cv2.imwrite(filename, common_pixel)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


True

In [54]:
img_folder = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\variance_edge"#variance_robust
common_pixel_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\common_pixel2.png"
output_decommoned_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\decommoned_edge_detect"

count = 0
for img_path in glob.glob(os.path.join(img_folder, '*.png')):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    common_pixel = cv2.imread(common_pixel_path, cv2.IMREAD_GRAYSCALE)
    decommoned_img = img-common_pixel
    filename = os.path.join(output_decommoned_path, f'decommoned_image_{count}.png')
    cv2.imwrite(filename, decommoned_img)
    count = count+1

#### 4.3 Find Local Intersections of Several Frames and Delete Fixed Noise

In [27]:
img_folder = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\selected_var_robust"
output_common_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2"
img_li = []

for img_path in glob.glob(os.path.join(img_folder, '*.png')):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_li.append(img)


def find_common_pixels(matrix_li):
    # stack into a 3D matrix
    stacked_matrices = np.stack(matrix_li, axis=0)  # shape is (41, 512, 512)
    
    # find common pixels
    common_pixels = np.all(stacked_matrices == stacked_matrices[0], axis=0)
    
    # create a new matrix to store common pixels, and mark 0 for other pixels
    result_matrix = np.where(common_pixels, stacked_matrices[0], 0)
    
    return result_matrix

common_pixel = find_common_pixels(img_li)
print(common_pixel)
filename = os.path.join(output_common_path, f'selected_common_pixel.png')
cv2.imwrite(filename, common_pixel)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


True

In [28]:
img_folder = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\selected_var_robust"
common_pixel_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\selected_common_pixel.png"
output_decommoned_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\selected_decommoned"

count = 0
for img_path in glob.glob(os.path.join(img_folder, '*.png')):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    common_pixel = cv2.imread(common_pixel_path, cv2.IMREAD_GRAYSCALE)
    decommoned_img = img-common_pixel
    filename = os.path.join(output_decommoned_path, f'decommoned_image_{count}.png')
    cv2.imwrite(filename, decommoned_img)
    count = count+1

### 5. Filters for Intense Frame

#### 5.1 Ridge Detection Filter

In [82]:
def ridge_filter(input_path, output_path):
    image = cv2.imread(input_path)
    ridge_filter = cv2.ximgproc.RidgeDetectionFilter_create()
    ridges = ridge_filter.getRidgeFilteredImage(image)
    
    filename = os.path.join(output_path, f'ridege_filter.png')
    plt.imsave(filename, ridges, cmap='gray')

In [83]:
m = 2 # a specific angle folder, e.g., "\\20250103102526.486000\\2\\demeaned_image"

input_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\intense_images\\intense_frame_22.png"
output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}"

ridge_filter(input_path, output_path)

#### 5.2 Hessian Filter

In [86]:
def hessian_filter(input_path, output_path, sigma=3.0):

    img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE).astype(np.int16)
    
    # calculate components of hessian
    hxx, hxy, hyy = hessian_matrix(img, sigma, use_gaussian_derivatives=False) 
    
    eigenvalues = hessian_matrix_eigvals((hxx, hxy, hyy))
    
    filename = os.path.join(output_path, f'hessian_filter_i1.png')
    plt.imsave(filename, eigenvalues[0], cmap='gray')

    filename = os.path.join(output_path, f'hessian_filter_i2.png')
    plt.imsave(filename, eigenvalues[1], cmap='gray')

In [87]:
m = 2 # a specific angle folder, e.g., "\\20250103102526.486000\\2\\demeaned_image"

input_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\intense_images\\intense_frame_22.png"
output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}"

hessian_filter(input_path, output_path)

#### 5.3 Sato Filter

In [88]:
def sato_filter(input_path, output_path):
    img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    sato_response = sato(img, sigmas=range(1, 15, 2), black_ridges=True, mode='reflect')
    smoothed_img = gaussian(sato_response, sigma=1)
    filename = os.path.join(output_path, f'sato_filter.png')
    plt.imsave(filename, smoothed_img, cmap='gray')

In [89]:
m = 2 # a specific angle folder, e.g., "\\20250103102526.486000\\2\\demeaned_image"

input_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\output\\28.png"
output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}"

sato_filter(input_path, output_path)

In [90]:
# thresholding for sato filter
input_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\sato_filter.png"
output_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2"

sato_img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
t = np.percentile(sato_img, 90)
sato_img[sato_img <= t] = 0
sato_img[sato_img > t] = 255
filename = os.path.join(output_path, f'sato_thres.png')
cv2.imwrite(filename, sato_img)

True

In [91]:
input_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\sato_thres.png"
var_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\variance_2.png"

variance = cv2.imread(var_path, cv2.IMREAD_GRAYSCALE)
img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)

t = np.percentile(variance, 75)
variance[variance <= t] = 0
variance[variance > t] = 1


new_img = np.multiply(img, variance)
        
output_path_new = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2"
filename = os.path.join(output_path_new, f'sato_var.png')
cv2.imwrite(filename, new_img)

True

#### 5.3 Sato Filter with Gray-Intense Image

Previousy we use mean (or background) of other patients to intensify image, here we use full gray image to intensify.

For mean, another idea is to choose a mean that has the greatest OT-distance from current mean, so that we can balance the light.

In [61]:
def gray_intense_img(gray_value, demeaned_path, output_path):
    
    img = cv2.imread(demeaned_path, cv2.IMREAD_GRAYSCALE).astype(np.int16)
    gray_img = np.full_like(img, gray_value, dtype=np.int16)
    result_img = img.astype(np.int16) + gray_img

    cv2.imwrite(output_path, result_img)

In [65]:
gray_value = 90
m = 4 # a specific angle folder, e.g., "\\20250103102526.486000\\2\\demeaned_image"
n = 36 # a specific best frame, e.g., "\\2\\demeaned_image\\32.png"

demeaned_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\demeaned_image\\{n}.png"#need to specify a best frame
output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\gray_intense_image.png"
gray_intense_img(gray_value, demeaned_path, output_path)

In [66]:
m = 2 # a specific angle folder, e.g., "\\20250103102526.486000\\2\\demeaned_image"

input_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\gray_intense_image.png"
output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}"

sato_filter(input_path, output_path)

#### 5.4 Robust Preprocess

In [53]:
def normalize_image(I):
    I = I.astype(np.float32)
    return (I - I.min()) / (I.max() - I.min())

def apply_threshold(I, T):
    return (I >= T).astype(np.uint8)

def compute_hessian_matrix(I, sigma):
    I_smooth = gaussian_filter(I, sigma)
    Ixx = cv2.Sobel(I_smooth, cv2.CV_64F, 2, 0, ksize=5)
    Iyy = cv2.Sobel(I_smooth, cv2.CV_64F, 0, 2, ksize=5)
    Ixy = cv2.Sobel(I_smooth, cv2.CV_64F, 1, 1, ksize=5)
    return Ixx, Iyy, Ixy

def compute_vesselness(I, scales, alpha=0.5, beta=0.5, tau=15):
    V = np.zeros_like(I, dtype=np.float32)

    for sigma in scales:
        Ixx, Iyy, Ixy = compute_hessian_matrix(I, sigma)
        
        # Compute eigenvalues of Hessian matrix
        eigenvalues = hessian_matrix_eigvals((Ixx, Iyy, Ixy))

        lambda1, lambda2 = eigenvalues[0], eigenvalues[1]

        # Compute vesselness measures
        BR = lambda1 / (lambda2 + 1e-10)  # Avoid division by zero
        S = np.sqrt(lambda1**2 + lambda2**2)

        # Compute vesselness response
        Vo = np.exp(-BR**2 / (2 * alpha**2)) * (1 - np.exp(-S**2 / (2 * beta**2)))
        Vo[lambda2 > 0] = 0  # Suppress non-tubular structures

        # Update vesselness map
        V = np.maximum(V, Vo)

    return V

def region_filling(V, vess_threshold, min_size):
    V_mask = (V >= vess_threshold).astype(np.uint8)

    # Identify connected components
    labeled, num_features = label(V_mask)

    # Remove small components
    for i in range(1, num_features + 1):
        if np.sum(labeled == i) < min_size:
            V_mask[labeled == i] = 0

    # Morphological closing to fill gaps
    V_mask_closed = binary_closing(V_mask)
    
    return V_mask_closed

def preprocess_vessel_image(I, scales, T, vess_threshold, min_size):
    I_norm = normalize_image(I)
    
    # Step 1: Global thresholding
    T_mask = apply_threshold(I_norm, T)

    # Step 2: Frangi vesselness filtering
    V = compute_vesselness(I_norm, scales)

    # Step 3: Region Filling
    V_mask = region_filling(V, vess_threshold, min_size)

    # Step 4: Combine threshold and vesselness maps
    C_mask = np.logical_or(T_mask, V_mask).astype(np.uint8)

    # Step 5: Final output
    S = np.logical_or(T_mask, C_mask).astype(np.uint8)
    
    return S

# Example usage
if __name__ == "__main__":
    # Load grayscale image
    
    for m in range(2,3):
        demeaned_path_0 = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\output"
        img_num = len(glob.glob(os.path.join(demeaned_path_0, '*.png')))  
        for n in range(img_num):
            input_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\output\\{n}.png"
            I = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            output_path = f"F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\{m}\\robust_preprocess"

            
            # Define parameters
            scales = np.arange(1, 20, 2)  # Scale range
            T = 0.5  # Global threshold
            vess_threshold = 0.4  # Vesselness threshold
            min_size = 5  # Minimum component size

            S = preprocess_vessel_image(I, scales, T, vess_threshold, min_size)
            S_new = 255-S
            filename = os.path.join(output_path, f'robust_preprocess_{n}.png')
            plt.imsave(filename, S_new, cmap='gray')

    # Display result
    #plt.figure(figsize=(12, 6))
    #plt.subplot(1, 2, 1)
    #plt.imshow(I, cmap="gray")
    #plt.title("Original Image")

    #plt.subplot(1, 2, 2)
    #plt.imshow(S, cmap="gray")
    #plt.title("Segmented Vessel Image")
    #plt.show()

### 6. Edge Detector

In [57]:
folder_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\output"
output_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\edge_detect"

count = 0

for img_path in glob.glob(os.path.join(folder_path, '*.png')):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    smoothed_img = gaussian(img, sigma=1.5)
    
    edges = filters.sobel(smoothed_img)
    filename = os.path.join(output_path, f'edge_detect_{count}.png')
    plt.imsave(filename, edges, cmap='gray')
    count = count+1

In [53]:
# thresholding for Edge Detector
input_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\edge_detect"
output_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\edge_detect_thres"

count = 0

for edges_path in glob.glob(os.path.join(input_path, '*.png')):
    edges = cv2.imread(edges_path, cv2.IMREAD_GRAYSCALE)
    t = np.percentile(edges, 90)
    edges[edges <= t] = 0
    edges[edges > t] = 255
    filename = os.path.join(output_path, f'edge_thres_{count}.png')
    cv2.imwrite(filename, edges)
    count = count+1

### 7. Fill-in by Gradients

In [4]:
folder_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\edge_detect"
for edges_path in glob.glob(os.path.join(folder_path, '*.png')):
    edges = cv2.imread(edges_path, cv2.IMREAD_GRAYSCALE)
    grad_x = cv2.Sobel(edges, cv2.CV_64F, 1, 0, ksize=3)  # horizontal gradient
    grad_y = cv2.Sobel(edges, cv2.CV_64F, 0, 1, ksize=3)  # vertical gradient
    grad_norm = np.sqrt(grad_x**2 + grad_y**2)  
    grad_direction = np.arctan2(grad_y, grad_x)
    
    #print(grad_y[:,:1])
    print(grad_direction)

[[ 0.          0.          3.14159265 ...  0.          3.14159265
   0.        ]
 [ 1.57079633  1.61424722  2.01521554 ...  1.52083793  1.89254688
   1.57079633]
 [ 1.57079633  1.64756822  2.40877755 ...  1.64756822  2.18152229
   1.57079633]
 ...
 [-1.57079633 -0.3470616  -3.08283683 ... -0.25732371 -0.83535656
  -1.57079633]
 [-1.57079633 -0.39707945 -1.79759517 ...  0.19739556 -1.62955215
  -1.57079633]
 [ 0.          0.          0.         ...  3.14159265  3.14159265
   0.        ]]
[[ 0.          0.          3.14159265 ...  3.14159265  3.14159265
   0.        ]
 [-1.57079633 -0.88506682 -1.30454428 ... -2.88426894 -3.08900959
   1.57079633]
 [-1.57079633 -0.29145679  0.29849893 ...  2.12939564  2.37944611
   1.57079633]
 ...
 [-1.57079633 -1.20089026 -1.94168762 ... -1.98320677 -0.78539816
  -1.57079633]
 [-1.57079633 -1.12701365 -1.57079633 ... -1.27933953 -0.78539816
  -1.57079633]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]
[[ 0.          0.       

In [55]:
def flood_fill(image, start_point):
    
    # mask with all black
    mask = np.zeros((image.shape[0] + 2, image.shape[1] + 2), dtype=np.uint8)
    
    # use flood fill to fill-in mask
    num, im, mask, rect = cv2.floodFill(image, mask, start_point, (255, 255, 255))
    return im


image_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\edge_detect_thres\\edge_thres_28.png"
output_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2"
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

grad_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
grad_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)

_, binary_image = cv2.threshold(grad_magnitude.astype(np.uint8), 50, 255, cv2.THRESH_BINARY)

for i in np.where(image[0,:]==255):
    start_point_y = i.tolist()[0]

start_point = (0,start_point_y) # the first white point in the first row is the vessel starting point

filled_image = flood_fill(binary_image, start_point)
filename = os.path.join(output_path, f'flood_fill.png')
cv2.imwrite(filename, filled_image)
#note that the fill-in should be white, it should not be that the vessel is white and blood is black

True

In [101]:
image_path = "F:\\JJU Dataset\\CAG without CCTA\\Huang Rui Cheng\\20250103102526.486000\\2\\edge_detect_thres\\edge_thres_28.png"
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
for i in np.where(image[0,:]==255):
    print(i.tolist())

[96, 97, 98, 104, 105, 106, 129, 130, 134, 262, 263, 272, 273, 277, 278, 279]


# 实验代码

In [18]:
dicom_path = "D:\\00200440\\Huang Rui Cheng\\Huang Rui Cheng\\20250103102526.486000\\1\\00572087_20250103_1_13.dcm"
output_folder = "D:\\00200440\\Huang Rui Cheng\\Huang Rui Cheng\\20250103102526.486000\\1\\output\\"

In [19]:
dicom_data = pydicom.dcmread(dicom_path)

In [20]:
pixel_array = dicom_data.pixel_array

In [21]:
for frame_index in range(pixel_array.shape[0]):
        frame = pixel_array[frame_index]
        if len(frame.shape) == 2:
            frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
        output_path = os.path.join(output_folder, f"frame_{frame_index:04d}.jpg")
        cv2.imwrite(output_path, frame)
        print(f"Saved {output_path}")

Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0000.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0001.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0002.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0003.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0004.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0005.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0006.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0007.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0008.jpg
Saved D:\00200440\Huang Rui Cheng\Huang Rui Cheng\20250103102526.486000\1\output\frame_0009.jpg
Saved D:\00200440\Huang Rui Cheng\Huang 